# くじびきラボ：Pythonで試す（v1.0.1）

くじを何度も引くと、ポイントはどう変わるでしょうか。設定を変えたり、同じ条件で繰り返したりして比べてみましょう。

2026年度3Q（月3木3クラス）の学生作品をもとにした学習版です。元作品の当選確率と配当を受け継いでいます。

**すぐ遊びたい方は[ブラウザ版](https://shogo-ishikawa.github.io/2026_3Q_M3Th3/projects/lottery-lab/)へ。**

## 最初の操作

1. セル1の ▶ を押します。追加インストールやGPUは不要です。
2. セル2で設定と回数を選び、▶ を押します。
3. 全設定を繰り返し比べたい場合は、セル3も実行します。

リンクからそのまま実行できます。変更したコードや設定を残したい場合だけ「ファイル → ドライブにコピーを保存」を選んでください。

[操作マニュアル](https://github.com/shogo-ishikawa/2026_3Q_M3Th3/blob/main/projects/lottery-lab/docs/manual.md) ／ [コードの読み方](https://github.com/shogo-ishikawa/2026_3Q_M3Th3/blob/main/projects/lottery-lab/docs/development.md)

## 1. 抽選プログラムを準備する

▶ を押してから、セル2へ進みます。

In [ ]:
#@title 1. 抽選プログラムを準備する { display-mode: "form" }
"""くじびきラボのPython学習版。標準ライブラリだけで抽選できます。"""
import random


# 元作品の6種類の設定と配当を受け継いでいます。
BONUS_PROBABILITIES = {
    1: {"BIG": 1 / 273.1, "REG": 1 / 439.8},
    2: {"BIG": 1 / 269.7, "REG": 1 / 399.6},
    3: {"BIG": 1 / 269.7, "REG": 1 / 331.0},
    4: {"BIG": 1 / 259.0, "REG": 1 / 315.1},
    5: {"BIG": 1 / 259.0, "REG": 1 / 255.0},
    6: {"BIG": 1 / 255.0, "REG": 1 / 255.0},
}
PAYOUTS = {
    "BIG": 252,
    "REG": 96,
    "ブドウ": 8,
    "チェリー": 2,
    "リプレイ": 3,
    "ベル": 14,
    "ピエロ": 10,
    "ハズレ": 0,
}


def check_integer(name, value, minimum, maximum):
    """入力が範囲内の整数か確認します。"""
    if isinstance(value, bool) or not isinstance(value, int) or not minimum <= value <= maximum:
        raise ValueError(f"{name}は{minimum}〜{maximum}の整数にしてください。")


def probabilities(setting):
    """設定番号から、役と当選確率のリストを作ります。"""
    check_integer("設定", setting, 1, 6)
    lottery = list(BONUS_PROBABILITIES[setting].items())
    grape_probability = 1 / 5.79 if setting == 6 else 1 / 6.05
    lottery.extend([
        ("ブドウ", grape_probability),
        ("チェリー", 1 / 35.6),
        ("リプレイ", 1 / 7.3),
        ("ベル", 1 / 1024),
        ("ピエロ", 1 / 1024),
    ])
    lottery.append(("ハズレ", 1 - sum(probability for _, probability in lottery)))
    return lottery


def pick_result(value, lottery):
    """0以上1未満の数を、累積確率で役に振り分けます。"""
    if not 0 <= value < 1:
        raise ValueError("乱数は0以上1未満にしてください。")
    cumulative = 0
    for name, probability in lottery:
        cumulative += probability
        if value < cumulative:
            return name
    return "ハズレ"


def expected_delta(setting, play_cost=3):
    """1回あたりの理論上の平均収支。1回ごとの結果とは異なります。"""
    check_integer("1回の消費ポイント", play_cost, 1, 100)
    return sum(probability * PAYOUTS[name] for name, probability in probabilities(setting)) - play_cost


def simulate(setting=6, games=1000, initial_points=50, play_cost=3, seed=42):
    """くじを指定回数引き、収支・回数・ポイント推移を返します。"""
    check_integer("回数", games, 1, 5000)
    check_integer("初期ポイント", initial_points, 0, 1_000_000)
    check_integer("1回の消費ポイント", play_cost, 1, 100)
    check_integer("乱数の種", seed, 0, 2**32 - 1)
    lottery = probabilities(setting)
    rng = random.Random(seed)
    balance = initial_points
    counts = {name: 0 for name in PAYOUTS}
    history = [balance]
    earned = 0

    for game in range(games):
        name = pick_result(rng.random(), lottery)
        payout = PAYOUTS[name]
        balance += payout - play_cost
        earned += payout
        counts[name] += 1
        history.append(balance)

    return {
        "setting": setting,
        "games": games,
        "initial_points": initial_points,
        "play_cost": play_cost,
        "seed": seed,
        "balance": balance,
        "spent": games * play_cost,
        "earned": earned,
        "net": balance - initial_points,
        "counts": counts,
        "history": history,
    }


def compare_settings(games=1000, runs=50, seed=42, initial_points=50, play_cost=3):
    """全設定を繰り返し、平均とばらつきを比べます。"""
    check_integer("繰り返し回数", runs, 1, 100)
    check_integer("回数", games, 1, 5000)
    check_integer("乱数の種", seed, 0, 2**32 - 1)
    comparison = []
    for setting in range(1, 7):
        net_results = []
        total_history = [0] * (games + 1)
        for run in range(runs):
            # 同じ試行番号では全設定に同じ乱数列を使います。
            result = simulate(setting, games, initial_points, play_cost, (seed + run) % 2**32)
            net_results.append(result["net"])
            for index, balance in enumerate(result["history"]):
                total_history[index] += balance
        comparison.append({
            "setting": setting,
            "mean_net": sum(net_results) / runs,
            "min_net": min(net_results),
            "max_net": max(net_results),
            "positive_rate": sum(net > 0 for net in net_results) / runs,
            "expected_net": expected_delta(setting, play_cost) * games,
            "net_results": net_results,
            "mean_history": [balance / runs for balance in total_history],
        })
    return comparison

print("準備ができました。セル2で設定を選びましょう。")


## 2. くじを引いてみる

最初はそのまま ▶ を押してみましょう。初期ポイントは50点、1回につき3点を使います。仮想ポイントなので、マイナスになっても指定回数まで続けます。

同じ乱数の種 `seed` なら、同じ結果を再現できます。違う結果を見たいときは `seed` を変えて再実行してください。
点線は確率から計算した平均です。1回の実験がこの線どおりになるわけではありません。

In [ ]:
#@title 2. くじを引いてみる { display-mode: "form" }
setting = 6 #@param [1, 2, 3, 4, 5, 6] {type:"raw"}
games = 1000 #@param {type:"integer"}
initial_points = 50 #@param {type:"integer"}
play_cost = 3 #@param {type:"integer"}
seed = 42 #@param {type:"integer"}

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    result = simulate(setting, games, initial_points, play_cost, seed)
except ValueError as error:
    print(error)
else:
    print(f"設定{setting}で{games}回：最終ポイント {result['balance']:,}点／収支 {result['net']:+,}点")
    print(f"消費 {result['spent']:,}点／獲得 {result['earned']:,}点")
    rows = ["| 役 | 当選回数 |", "| --- | ---: |"]
    for name, count in result["counts"].items():
        rows.append(f"| {name} | {count} |")
    display(Markdown("\n".join(rows)))
    expected = [initial_points + game * expected_delta(setting, play_cost) for game in range(games + 1)]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(result["history"], label="This run", color="#28645b")
    ax.plot(expected, linestyle="--", label="Expected average", color="#c0783d")
    ax.axhline(initial_points, color="#bbb", linewidth=0.8)
    ax.set(xlabel="Draws", ylabel="Points", title=f"Setting {setting}")
    ax.grid(alpha=0.2)
    ax.legend()
    plt.show()


## 3. 全設定を繰り返し比べる

1000回のくじ引きを50回ずつ繰り返し、平均収支と結果の幅を比べます。`comparison_games` は1試行の抽選回数、`repeat_runs` は試行数です。

同じ試行番号では全設定に同じ乱数列を使います。確率の違いを比べやすくするためです。
表の最低・最高は今回の試行で出た値です。今後の結果がこの範囲に収まる保証ではありません。

ブラウザ版とPython版では乱数の作り方が異なるため、同じseedでも結果は一致しません。それぞれの版の中では再現できます。

In [ ]:
#@title 3. 全設定を繰り返し比べる { display-mode: "form" }
comparison_games = 1000 #@param {type:"integer"}
repeat_runs = 50 #@param {type:"integer"}
comparison_seed = 42 #@param {type:"integer"}

import matplotlib.pyplot as plt
from IPython.display import display, Markdown

try:
    comparison = compare_settings(games=comparison_games, runs=repeat_runs,
                                 seed=comparison_seed, initial_points=50, play_cost=3)
except ValueError as error:
    print(error)
else:
    print(f"各設定：{comparison_games:,}回のくじ引きを{repeat_runs}試行。初期50点、1回3点です。")
    rows = ["| 設定 | 平均収支 | 最低 | 最高 | プラスの割合 | 理論上の平均収支 |",
            "| --- | ---: | ---: | ---: | ---: | ---: |"]
    for row in comparison:
        rows.append(f"| {row['setting']} | {row['mean_net']:+.1f} | {row['min_net']:+,} | {row['max_net']:+,} | {row['positive_rate']:.0%} | {row['expected_net']:+.1f} |")
    display(Markdown("\n".join(rows)))
    fig, ax = plt.subplots(figsize=(10, 4))
    for row in comparison:
        ax.plot(row["mean_history"], label=f"Setting {row['setting']}")
    ax.set(xlabel="Draws", ylabel="Average points", title=f"Average of {repeat_runs} runs")
    ax.grid(alpha=0.2)
    ax.legend(ncol=3)
    plt.show()


## 仕組みを変えてみる

セル1のコードを開くと、当選確率の辞書 `BONUS_PROBABILITIES` と配当の辞書 `PAYOUTS` が見つかります。例えばBIGの配当を変えると、ポイント推移と平均収支はどう変わるでしょうか。

変更後はセル1から実行し直します。確率の合計は1以下にし、残りをハズレにする考え方を守ってください。
元の結果と比べたい場合は、変更前後で同じseedを使ってみましょう。